# **Multi Layer Stacking**

In [17]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import accuracy_score, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

import warnings
warnings.filterwarnings('ignore') 

In [18]:
from sklearn.datasets import make_regression

X, y = make_regression(
    n_samples=1000,
    n_features=6,
    noise=0.2,
    random_state=42
)

X = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(6)])
y = pd.DataFrame(y, columns=['target'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=8)


X_train.head()

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5
578,-1.724918,-0.562288,-1.012831,0.241962,-1.913280,0.314247
66,-0.305189,1.046912,-0.335244,-0.138512,-0.423890,-1.341065
39,-0.245743,-0.272724,-2.696887,-0.259591,-1.503143,-0.054295
302,-0.921860,-1.003957,0.207267,-0.083438,-1.449645,0.069344
974,-0.936506,-0.667780,0.292193,-0.031059,-0.909683,-0.187329


In [ ]:
# multi layer stacking

from sklearn.ensemble import StackingRegressor

lvl0_models = [
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('knn', KNeighborsRegressor(n_neighbors=10)),
    ('gbt', GradientBoostingRegressor())
]

# taking different models for lvl 1
lvl1_models_1 = StackingRegressor(
    estimators=lvl0_models,
    final_estimator=Ridge(alpha=1),
    cv=5,
    n_jobs=-1
)

lvl1_models_2 = StackingRegressor(
   estimators=lvl0_models,
    final_estimator=Lasso(),
    cv=5,
    n_jobs=-1
)

from sklearn.svm import SVR
lvl1_models_3 = StackingRegressor(
    estimators=lvl0_models,
    final_estimator=SVR(kernel='rbf'),
    cv=5,
    n_jobs=-1
)


# putting all level 1 models in one object
all_lvl1_models=[
    ('model_1', lvl1_models_1),
    ('model_2', lvl1_models_2),
    ('model_3', lvl1_models_3)
]


# making the final meta model
meta = StackingRegressor(
    estimators=all_lvl1_models,
    final_estimator=LinearRegression(),
    cv=5,       # remember to avoid the heavy computations, you need to set cv to 0, coz by default the cv is set to 5, even if you dont mention it in the code.
    n_jobs=-1
)



In [20]:
# training the model
meta.fit(X_train, y_train)
y_pred =meta.predict(X_test)


In [21]:
# making predictions

print(f"ACCURACY : {np.round(r2_score(y_test, y_pred),2)}")



ACCURACY : 0.97
